# Clustering & Anomaly Detection

> ####  List of exercises
>   - **Exercise 1**: Clustering Handwritten Digits with K-Means and DBSCAN
>   - **Exercise 2**: Online Retail Customer Segmentation

## Load Dataset

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets

In [0]:
digits = datasets.load_digits()

## Preview Data

In [0]:
type(digits)

In [0]:
print(digits.keys())

In [0]:
digits['DESCR']

In [0]:
digits['feature_names']

In [0]:
digits['data'][:5]

In [0]:
digits['target_names']

In [0]:
# Class distribution in target variable
np.unique(digits['target'], return_counts=True)

In [0]:
np.unique(digits['target'])

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [0]:
# Number of classes
classes = np.unique(digits['target'])
n_classes = len(classes)

# Store colors for plots
colors = plt.cm.tab10.colors

# Plot setup
fig, ax = plt.subplots(figsize=(8, 6))

X_orig = digits['data']
y = digits['target']

pca = PCA(n_components=2)
X = pca.fit_transform(StandardScaler().fit_transform(X_orig))

ix, iy = 0, 1

for i, digit in enumerate(classes):
    idx = y == classes[i]
    ax.scatter(X[idx,ix], 
               X[idx,iy],
               color=colors[i],
               marker='o',
               label=digits['target_names'][i])
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('PC1')    
plt.ylabel('PC2') 
plt.tight_layout()

We'll use the 2D principal components of the **handwritten digits dataset** as an "unlabelled" dataset to illustrate various clustering approaches in the sections that follow.

# K-Means Clustering

## How it works:

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


# Step 1: Initialize KMeans parameters
K = 3
# np.random.seed(42)
initial_centroids = X[np.random.choice(len(X), K, replace=False)]
centroids = initial_centroids.copy()
labels = np.zeros(len(X))
max_iters = 10
colors = plt.cm.tab10.colors

# Step 2: Store centroid and label history for animation
centroid_history = [centroids.copy()]
label_history = []

def assign_clusters(X, centroids):
    distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
    return np.argmin(distances, axis=1)

def update_centroids(X, labels, K):
    return np.array([X[labels == k].mean(axis=0) if np.any(labels == k) else np.random.rand(2)*10 for k in range(K)])

for _ in range(max_iters):
    labels = assign_clusters(X, centroids)
    label_history.append(labels.copy())
    centroids = update_centroids(X, labels, K)
    centroid_history.append(centroids.copy())

# Step 3: Create the animation

def update(frame):
    ax.clear()
    ax.set_title(f'K-Means Iteration {frame}', fontsize=14)
    labels = label_history[frame]
    centroids = centroid_history[frame]

    for k in range(K):
        ax.scatter(X[labels == k, 0], X[labels == k, 1], color=colors[k], label=f'Cluster {k}', s=30)
    ax.scatter(centroids[:, 0], centroids[:, 1], c='black', marker='+', s=100, label='Centroids')
    ax.set_xlim(X[:, 0].min() - 1, X[:, 0].max() + 1)
    ax.set_ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# plt.legend(bbox_to_anchor=(1.05, 1.0), loc='upper left')

fig, ax = plt.subplots(figsize=(10, 6))
fig.subplots_adjust(right=0.75)  # Reserve space on the right for the legend

ani = FuncAnimation(fig, update, frames=max_iters, interval=1000, repeat=False);
plt.close(fig)  # Prevents duplicate plot rendering in Jupyter
HTML(ani.to_jshtml())

## Deciding on an optimal K
### Scree Plot and Silhoutte Score

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs, make_moons

import ipywidgets as widgets
from IPython.display import display, clear_output

# Inline plotting
%matplotlib inline

# Generate data
X, _ = make_blobs(n_samples=500, centers=4, cluster_std=0.7, random_state=42)

# Widgets
max_k_slider = widgets.IntSlider(
    value=10, min=3, max=15, step=1,
    description='Max K:', continuous_update=False
)

output = widgets.Output()

def plot_kmeans_metrics(max_k):
    k_values = range(2, max_k + 1)
    inertias = []
    silhouette_scores = []

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(X)
        inertias.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(X, labels))

    fig, axs = plt.subplots(1, 2, figsize=(12, 5))

    axs[0].plot(k_values, inertias, marker='o')
    axs[0].set_title("Elbow Method")
    axs[0].set_xlabel("Number of Clusters (K)")
    axs[0].set_ylabel("Inertia")
    axs[0].grid(True)

    axs[1].plot(k_values, silhouette_scores, marker='o', color='orange')
    axs[1].set_title("Silhouette Score")
    axs[1].set_xlabel("Number of Clusters (K)")
    axs[1].set_ylabel("Score")
    axs[1].grid(True)

    plt.tight_layout()
    return fig

# Event handler
def update_plot(change):
    with output:
        output.clear_output(wait=True)
        fig = plot_kmeans_metrics(max_k_slider.value)
        display(fig)
        plt.close(fig)

# Initial plot
max_k_slider.observe(update_plot, names='value')

# Display UI
display(widgets.VBox([max_k_slider, output]))

# Trigger first plot
update_plot(None)

### K-Means Clustering: Data with non-spherical clusters

In [0]:
# Generate a dataset with non-spherical clusters
X, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
X = StandardScaler().fit_transform(X)


# Step 2: Initialize KMeans parameters
K = 2
# np.random.seed(42)
initial_centroids = X[np.random.choice(len(X), K, replace=False)]
centroids = initial_centroids.copy()
labels = np.zeros(len(X))
max_iters = 10
# colors = ['r', 'b']
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, K)]

# Step 3: Store centroid and label history for animation
centroid_history = [centroids.copy()]
label_history = []

for _ in range(max_iters):
    labels = assign_clusters(X, centroids)
    label_history.append(labels.copy())
    centroids = update_centroids(X, labels, K)
    centroid_history.append(centroids.copy())

# Step 4: Create the animation
fig, ax = plt.subplots(figsize=(10, 6))
fig.subplots_adjust(right=0.75)  # Reserve space on the right for the legend


ani = FuncAnimation(fig, update, frames=max_iters, interval=1000, repeat=False);
plt.close(fig)  # Prevents duplicate plot rendering in Jupyter
HTML(ani.to_jshtml())


## DBSCAN

In [0]:
# What's Happening:
# We're using make_moons() to generate non-circular clusters (where K-Means would fail).

# DBSCAN clusters based on density and distance, so it can separate the moon shapes and detect outliers.

# Noise points are shown in black.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# %matplotlib inline (for Jupyter Notebooks)
%matplotlib inline

# Generate a dataset with non-spherical clusters
X, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
X = StandardScaler().fit_transform(X)

# Run DBSCAN
dbscan = DBSCAN(eps=0.3, min_samples=5)
labels = dbscan.fit_predict(X)

# Number of clusters (excluding noise)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

# Plotting the clusters
plt.figure(figsize=(8, 6))
unique_labels = set(labels)
colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]

for k, col in zip(unique_labels, colors):
    if k == -1:
        # Black used for noise.
        col = [0, 0, 0, 1]

    class_member_mask = (labels == k)
    xy = X[class_member_mask]
    plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
             markeredgecolor='k', markersize=8)

plt.title(f'DBSCAN Clustering\nClusters: {n_clusters}, Noise Points: {n_noise}')
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(False)
plt.show()

## Hierarchical Clustering

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.metrics import silhouette_score

# Generate sample data
X, _ = make_blobs(n_samples=150, centers=4, cluster_std=0.60, random_state=42)
X = StandardScaler().fit_transform(X)

# Compute the linkage matrix
linked = linkage(X, method='ward')  # Options: 'ward', 'single', 'complete', 'average'

# Plot dendrogram
plt.figure(figsize=(12, 6))
dendrogram(linked,
           orientation='top',
           distance_sort='descending',
           show_leaf_counts=False)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Sample index')
plt.ylabel('Distance')
plt.grid(True)
plt.show()

In [0]:
# Cut the dendrogram at a specific number of clusters (e.g., 4)
num_clusters = 4
labels = fcluster(linked, num_clusters, criterion='maxclust')

# Visualize clusters
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='Accent', s=50, edgecolor='k')
plt.title(f'Clusters from Hierarchical Clustering (k = {num_clusters})')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True)
plt.show()

# Evaluate with silhouette score
score = silhouette_score(X, labels)
print(f"Silhouette Score for {num_clusters} clusters: {score:.3f}")

# Anomaly Detection

## Clustering-Based Anomaly Detection

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler

# Create synthetic 2D data with some outliers
X, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.60, random_state=42)
outliers = np.random.uniform(low=-10, high=10, size=(20, 2))
X = np.vstack([X, outliers])
X = StandardScaler().fit_transform(X)

### Anomaly Detection with K-Means

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Fit KMeans
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X)

# Compute distances to centroids
distances = np.linalg.norm(X - kmeans.cluster_centers_[kmeans.labels_], axis=1)
threshold = np.percentile(distances, 95)
kmeans_outliers = distances > threshold
kmeans_inliers = ~kmeans_outliers

# Create a mesh grid
h = 0.01  # step size in the mesh
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predict cluster index for each point in the mesh
Z = kmeans.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plotting
plt.figure(figsize=(8, 6))

# Plot decision boundary by assigning a color to each region
plt.contourf(xx, yy, Z, alpha=0.2, cmap='Paired')

# Plot inliers and outliers
plt.scatter(X[kmeans_inliers, 0], X[kmeans_inliers, 1], c='skyblue', edgecolor='k', label='Inliers', s=30)
plt.scatter(X[kmeans_outliers, 0], X[kmeans_outliers, 1], color='black', label='Outliers', s=30)

# Plot centroids
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker='o', s=200, c='red', edgecolor='k', label='Centroids')

plt.title("K-Means Based Anomaly Detection with Decision Boundaries")
plt.legend()
plt.grid(True)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.show()

### Anomaly Detection with DBSCAN

In [0]:
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull
import numpy as np

# Fit DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(X)

# Plot DBSCAN results
plt.figure(figsize=(8, 6))
unique_labels = set(labels)

colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

for k, col in zip(unique_labels, colors):
    if k == -1:
        # Outliers
        col = 'black'
        marker = 'x'
        label = 'Outliers'
        edge_kws = {}
    else:
        marker = 'o'
        label = f'Cluster {k}'
        edge_kws = {'edgecolor': 'k'}

    class_member_mask = (labels == k)
    cluster_points = X[class_member_mask]

    plt.scatter(
        cluster_points[:, 0], cluster_points[:, 1],
        c=[col], marker=marker, s=30, label=label, **edge_kws
    )

    # Add convex hull boundary for non-outlier clusters
    if k != -1 and len(cluster_points) >= 3:
        hull = ConvexHull(cluster_points)
        for simplex in hull.simplices:
            plt.plot(cluster_points[simplex, 0], cluster_points[simplex, 1], c=col)

plt.title("DBSCAN Clusters with Approximate Boundaries")
plt.legend()
plt.grid(True)
plt.show()

## ML-Based Anomaly Detection (Isolation Forest & One-Class SVM)

In [0]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

# Create a meshgrid over the input space
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 500),
    np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 500)
)
grid = np.c_[xx.ravel(), yy.ravel()]

# Get the decision function for each point in the grid
Z = iso_forest.decision_function(grid)
Z = Z.reshape(xx.shape)

# Plotting
plt.figure(figsize=(8, 6))
# Contour plot for the decision boundary (zero level)
plt.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.PuBu)
plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='darkred')  # Decision boundary

# Scatter plot of the data points
plt.scatter(X[iso_preds == 1][:, 0], X[iso_preds == 1][:, 1], c='blue', edgecolor='k', s=30, label='Inliers')
plt.scatter(X[iso_preds == -1][:, 0], X[iso_preds == -1][:, 1], c='red', edgecolor='k', s=30, label='Outliers')

plt.title("Isolation Forest Anomaly Detection (with Decision Boundary)")
plt.legend()
plt.grid(True)
plt.show()

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import OneClassSVM

# --- ONE-CLASS SVM ---
oc_svm = OneClassSVM(kernel='rbf', nu=0.05, gamma='auto')
oc_preds = oc_svm.fit_predict(X)

# Create a meshgrid over the input space
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 500),
    np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 500)
)
grid = np.c_[xx.ravel(), yy.ravel()]

# Get the decision function for each point in the grid
Z = oc_svm.decision_function(grid)
Z = Z.reshape(xx.shape)

# Plotting
plt.figure(figsize=(8, 6))
# Contour plot for the decision boundary (zero level)
plt.contourf(xx, yy, Z, levels=np.linspace(Z.min(), 0, 7), cmap=plt.cm.PuBu)
plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='darkred')  # Decision boundary

# Scatter plot of the data points
plt.scatter(X[oc_preds == 1][:, 0], X[oc_preds == 1][:, 1], c='blue', edgecolor='k', s=30, label='Inliers')
plt.scatter(X[oc_preds == -1][:, 0], X[oc_preds == -1][:, 1], c='red', edgecolor='k', s=30, label='Outliers')

plt.title("One-Class SVM Anomaly Detection (with Decision Boundary)")
plt.legend()
plt.grid(True)
plt.show()

<br>
<div class="alert alert-info">
  <b><span style="font-size: 25px;">Exercise 1: Clustering Handwritten Digits with K-Means and DBSCAN</span></b>
</div>

### Objective:
Use unsupervised clustering algorithms (K-Means and DBSCAN) to group handwritten digit images from the UCI ML Digits dataset and evaluate their performance.
The dataset will be reduced to 10 dimensions using PCA to make clustering more effective.

### Instructions:
1. Load the digits dataset and reduce to 10 features using the following code:

    ```python
    pca = PCA(n_components=10, random_state=42)
    X_pca = pca.fit_transform(X) 
<br>  

2. Use K-Means to cluster the data into 10 groups.

3. Using the scree (elbow) and the silhouette score plots, what value of K (number of clusters) appears to be the most appropriate? Briefly justify your answer using both plots. 

4. Use DBSCAN to cluster the same data.

5. Evaluate both algorithms using:

    - Adjusted Rand Index (ARI)

    - Normalized Mutual Information (NMI)

    - Silhouette Score

6. Visualize the clustering results in 2D using the first two principal components.

7. Reflect:

   7.1 How do K-Means and DBSCAN perform differently?

   7.2 Rerun using agglomerative clustering

<br>
<div class="alert alert-info">
  <b style="font-size: 25px;">Your Solution Here</b>
</div>

In [0]:
# Step 0: Load libraries

In [0]:
# Step 1: Load and reduce dimensionality

In [0]:
# Step 2: Apply K-Means (using = 10)

In [0]:
# Step 3: Find optimal value of K using the Elbow and Silhoutte plots

In [0]:
# Step 4: Apply DBSCAN 

In [0]:
# Step 5: Evaluation
# An evaluation function you can use has been defined below

def evaluate_clustering(y_true, y_pred, name):
    unique_labels = np.unique(y_pred)
    if len(unique_labels) <= 1:
        print(f"\n{name} Clustering Evaluation:")
        print("Only one cluster found — skipping evaluation.")
        return
    print(f"\n{name} Clustering Evaluation:")
    print(f"ARI:  {adjusted_rand_score(y_true, y_pred):.3f}")
    print(f"NMI:  {normalized_mutual_info_score(y_true, y_pred):.3f}")
    print(f"Silhouette Score: {silhouette_score(X_pca, y_pred):.3f}")



In [0]:
# Step 6: Visualize the clustering results
# Hints:
#   1. To compare visually use first 2 principal components of the X data in PCA space
#   2. Add different for each cluster to each plot. Note that there is  correlation 
#      between clusters in PCA and DBSCAN, and each color must be interpreted within 
#      the context of each separate plot



<br>
<div class="alert alert-info">
<b style="font-size: 25px;">Exercise 1: The End</b>
</div>

<br>
<div class="alert alert-info">
  <b><span style="font-size: 25px;">Exercise 2: Online Retail Customer Segmentation</span></b>
</div>

### Objective:
Apply RFM analysis and unsupervised learning algorithms (K-Means, DBSCAN) to segment customers from the UCI Online Retail dataset, and interpret meaningful patterns among segments for potential marketing strategies.

### Dataset
- **Source**: [UCI ML Repository](https://archive.ics.uci.edu/dataset/352/online+retail)
- **Description**: This is a transactional data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.The company mainly sells unique all-occasion gifts. Many customers of the company are wholesalers.
- **Reference Paper**: Daqing Chen, Sai Laing Sain, Kun Guo (2012). *Data mining for the online retail industry: A case study of RFM model-based customer segmentation using data mining*. Journal of Database Marketing and Customer Strategy Management, Vol. 19, No. 3.

### Instructions
#### **Part 1: Data Cleaning and Preparation**
- Load the dataset using the `fetch_ucirepo`.

  We're interested in the ids and features components only. Use `pd.concat` to gather these into a dataframe
- Filter for entries with:
    - Non-null `CustomerID`
      
    - `Quantity` > 0 and `UnitPrice` > 0
 
    - `Country` == "United Kingdom" (for simplicity)

- Convert `InvoiceDate` to datetime format.

#### **Part 2: RFM Feature Engineering**
- Define a snapshot date as the day after the last invoice.

- Compute:

   - Recency = Days since last purchase

   - Frequency = Number of unique invoices

   - Monetary = Total spend (Quantity × UnitPrice)

#### **Part 3: Exploratory Analysis**
- Visualize histograms/distributions of RFM features.

- Scale the RFM features using `StandardScaler`.

#### **Part 4: Clustering**
1. K-Means:
   - Use the Elbow method and Silhouette Score to determine the optimal number of clusters.

   - Visualize cluster centers and RFM distribution per cluster.
 
   - Use PCA or t-SNE for 2D visualization of clusters.

2. DBSCAN:
   - Cluster with DBSCAN (adjust `eps` and `min_samples`).

   - Compare with K-Means results and analyze outliers.
 
#### **Part 5: Interpretation**
- Label clusters with descriptive names (e.g., "High Value", "At Risk", "Occasional").

- Reflect on potential marketing strategies for each cluster.

#### **Part 6: Additional Analysis**
- Filter for multiple countries and compare clustering.

<br>
<div class="alert alert-info">
  <b style="font-size: 25px;">Your Solution Here</b>
</div>

<br>
<div class="alert alert-info">
<b style="font-size: 25px;">Exercise 2: The End</b>
</div>